# exp105_compact_rank_slot_features_on_exp098 inference

Run saved lgb1 fold boosters on raw-test full replay features plus target-free selector rank-slot features.

## Contents

1. Setup and configuration
2. Input and model plan
3. Saved model inference
4. Metrics and submission preview

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from compact_rank_slot_features_on_exp098 import find_model_manifest, run_saved_model_inference
from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Inference mode:", get_nested(config, "inference.mode"))
print("Selected variant:", get_nested(config, "inference.selected_variant"))
print("Selected mode:", get_nested(config, "inference.selected_mode"))
print("Selected model:", get_nested(config, "inference.selected_model"))
print("Kaggle sources:", get_nested(config, "runtime.kaggle.inference_kernel_sources"))
print("Artifacts:", paths.artifacts_dir)
print("Submission:", paths.submission_path)

if get_nested(config, "inference.mode") == "not_selected_train_side_audit_only":
    raise SystemExit("Inference is intentionally disabled until train-side compact OOF is reviewed.")

## 2. Input and model plan

In [ ]:
sample = pd.read_csv(paths.sample_submission_path, dtype={"id": str})
manifest_path = find_model_manifest(get_nested(config, "inference.model_manifest_path"))
manifest = json.loads(Path(manifest_path).read_text())
selected_variant = get_nested(config, "inference.selected_variant") or "compact_rank_slot_features"
selected_mode = get_nested(config, "inference.selected_mode") or "gpu_repro_guard_dp_threads8"
selected_model = get_nested(config, "inference.selected_model") or "lgb1"
selected_models = [
    row
    for row in manifest.get("models", [])
    if row.get("variant") == selected_variant
    and row.get("mode") == selected_mode
    and row.get("model") == selected_model
]

print("Sample submission shape:", sample.shape)
print("Manifest:", manifest_path)
print("Manifest experiment:", manifest.get("experiment"))
print("Available variants:", [row.get("name") for row in manifest.get("variants", [])])
print("Selected saved boosters:", len(selected_models))
print("Base feature count:", len(manifest.get("feature_source", {}).get("feature_columns", [])))
print("Rank-slot groups:", {k: len(v) for k, v in manifest.get("rank_slot_feature_groups", {}).items()})
sample.head()

## 3. Saved model inference

In [ ]:
summary = run_saved_model_inference(
    output_dir=paths.artifacts_dir,
    submission_path=paths.submission_path,
    sample_submission_path=paths.sample_submission_path,
    data_dir=paths.raw_data_dir,
    test_dir=paths.test_data_dir,
    model_manifest_path=get_nested(config, "inference.model_manifest_path"),
    rank_slot_config=get_nested(config, "model.rank_slot"),
    variant_name=selected_variant,
    mode_name=selected_mode,
    model_name=selected_model,
    submission_target_column=get_nested(config, "data.submission_target_column") or "tvt",
    n_jobs=get_nested(config, "runtime.num_workers"),
    fast=bool(get_nested(config, "audit.fast")),
    use_gpu="auto",
)
summary

## 4. Metrics and submission preview

In [ ]:
metrics_path = paths.metrics_path
metrics_path.write_text(json.dumps(summary, indent=2))
metrics = pd.DataFrame([summary["metrics"]])
submission = pd.read_csv(paths.submission_path)

print("Wrote metrics:", metrics_path)
print("Wrote submission:", paths.submission_path)
display(metrics)
display(submission.head())
display(submission.describe(include="all"))